In [54]:
from google.colab import files
files.upload()

Saving kaggle.json to kaggle (8).json


{'kaggle (8).json': b'{"username":"yashvardhanpatil6001","key":"KGAT_c28bbf0f08e784a49a97bac88b50844b"}'}

In [55]:
!mkdir -p /root/.kaggle
!cp kaggle.json /root/.kaggle/kaggle.json
!chmod 600 /root/.kaggle/kaggle.json

In [57]:
!cat kaggle.json

{
"username":"Yashvardhan_Patil_6001",
"key":"KGAT_3494bf288f56bc8784682f1f47330148"
}

In [59]:
!kaggle datasets download -d karakaggle/kaggle-cat-vs-dog-dataset

Dataset URL: https://www.kaggle.com/datasets/karakaggle/kaggle-cat-vs-dog-dataset
License(s): unknown
100% 787M/787M [00:10<00:00, 79.0MB/s]



In [60]:
import zipfile

with zipfile.ZipFile('kaggle-cat-vs-dog-dataset.zip', 'r') as zip_ref:
    zip_ref.extractall('/content/')

In [61]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

from torch.utils.data import random_split

In [64]:
import os
print(os.listdir('/content/'))

['.config', 'kaggle (4).json', 'kaggle (7).json', 'kaggle (2).json', 'Kaggle (1).json', 'kaggle (3).json', 'kaggle.json', 'kaggle (1).json', 'kaggle (5).json', 'Kaggle.json', 'kaggle (6).json', 'kaggle-cat-vs-dog-dataset.zip', 'kaggle (8).json', 'kagglecatsanddogs_3367a', 'sample_data']


In [66]:
print(os.listdir('/content/kagglecatsanddogs_3367a'))

['readme[1].txt', 'MSR-LA - 3467.docx', 'PetImages']


In [62]:
from PIL import ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True

In [67]:
transform=transforms.Compose([
    transforms.Resize((32, 32)),
    transforms.ToTensor(),
    transforms.Normalize((0.5,0.5,0.5),(0.5,0.5,0.5))
])



dataset = datasets.ImageFolder(root='/content/kagglecatsanddogs_3367a/PetImages', transform=transform)

train_size = int(0.8 * len(dataset))
test_size  = len(dataset) - train_size

train_dataset, test_dataset = random_split(dataset, [train_size, test_size])

train_loader = DataLoader(train_dataset,batch_size=64,shuffle=True)

test_loader = DataLoader(test_dataset,batch_size=64,shuffle=False)


In [68]:
class CNN(nn.Module):
    def __init__(self):
        super(CNN,self).__init__()

        self.conv_layers=nn.Sequential(
        nn.Conv2d(3,32,kernel_size=3,padding=1),
        nn.ReLU(),
        nn.MaxPool2d(2,2),

        nn.Conv2d(32,64,kernel_size=3,padding=1),
        nn.ReLU(),
        nn.MaxPool2d(2,2),

        nn.Conv2d(64,128,kernel_size=3,padding=1),
        nn.ReLU(),
        nn.MaxPool2d(2,2)
        )

        self.fc_layers=nn.Sequential(
            nn.Linear(4*4*128,256),
            nn.ReLU(),

            nn.Linear(256,2)
        )

    def forward(self,x):
        x=self.conv_layers(x)
        x=x.view(x.size(0),-1)
        x=self.fc_layers(x)

        return x











In [69]:
model=CNN()
criterion=nn.CrossEntropyLoss()
optimizer=optim.Adam(model.parameters())


In [71]:
epochs=8

for epoch in range(epochs):

    epoch_training_loss=0.0
    for image,labels in train_loader:
        optimizer.zero_grad()

        output=model.forward(image)
        loss=criterion(output,labels)
        loss.backward()
        optimizer.step()
        epoch_training_loss+=loss.item()

    print(f"{epoch+1}/{epochs} and loss={epoch_training_loss/len(train_loader)}")





1/8 and loss=0.5499687287478875
2/8 and loss=0.47586398638593846
3/8 and loss=0.41803957039538103
4/8 and loss=0.3706853050165452
5/8 and loss=0.321073708196099
6/8 and loss=0.2781044531804629
7/8 and loss=0.23006484526185653
8/8 and loss=0.18957939505195007


In [73]:
correct_labels=0
total_labels=0

model.eval()

with torch.no_grad():
    for images,labels in test_loader:
        outputs=model.forward(images)
        _,predicted=torch.max(outputs,1)

        correct_labels+=(predicted==labels).sum().item()
        total_labels+=labels.size(0)
print(f"Accuracy={correct_labels/total_labels*100}")


/usr/local/lib/python3.12/dist-packages/PIL/TiffImagePlugin.py:950: UserWarning: Truncated File Read
  warnings.warn(str(msg))


Accuracy=82.13141025641025
